In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import talib
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os

print("All libraries imported successfully!")


All libraries imported successfully!


Cell 2: Load My Cleaned Data

In [4]:
# Set the correct file path
file_path = r"D:\Personal\KAIM-10 Academy\Week 1\Project Work\Price_Prediction_Week_1_Challenge\data\processed\cleaned_stock_data.csv"

# Load the merged cleaned data
try:
    merged_data = pd.read_csv(file_path)
    print(f"✅ Successfully loaded data: {len(merged_data)} rows")
    print(f"Columns: {list(merged_data.columns)}")
except FileNotFoundError:
    print(f"❌ File not found at: {file_path}")
    # Alternative: check current directory
    print("Checking current directory for file...")
    if os.path.exists("cleaned_stock_data.csv"):
        merged_data = pd.read_csv("cleaned_stock_data.csv")
        print(f"✅ Loaded from current directory: {len(merged_data)} rows")
    else:
        raise FileNotFoundError("Could not find cleaned_stock_data.csv")

# Display basic info
print(f"\nData shape: {merged_data.shape}")
print(f"Columns: {list(merged_data.columns)}")
print(f"\nFirst few rows:")
print(merged_data.head())

✅ Successfully loaded data: 1007 rows
Columns: ['Date', 'Close', 'High', 'Low', 'Open', 'Volume']

Data shape: (1007, 6)
Columns: ['Date', 'Close', 'High', 'Low', 'Open', 'Volume']

First few rows:
         Date              Close               High                Low  \
0         NaN               AAPL               AAPL               AAPL   
1  2020-01-02  72.46825408935547  72.52857392672578  71.22325176244068   
2  2020-01-03  71.76372528076172  72.52375386627762  71.53933722124737   
3  2020-01-06   72.3355484008789  72.37415398165305   70.6345318765685   
4  2020-01-07  71.99536895751953  72.60097528872254  71.77580380389517   

                Open     Volume  
0               AAPL       AAPL  
1  71.47659213409148  135480400  
2  71.69616734789946  146322800  
3  70.88546446996966  118387200  
4  72.34521969510813  108872000  


Cell 3: Check Data Structure and Symbols

In [5]:
# Check what symbols we have in the data
if 'Symbol' in merged_data.columns:
    symbols = merged_data['Symbol'].unique()
    print(f"Stock symbols found: {list(symbols)}")
    print(f"Number of unique stocks: {len(symbols)}")
    
    # Check date range for each symbol
    for symbol in symbols:
        symbol_data = merged_data[merged_data['Symbol'] == symbol]
        print(f"{symbol}: {len(symbol_data)} rows, from {symbol_data['Date'].min()} to {symbol_data['Date'].max()}")
else:
    print("❌ 'Symbol' column not found in data")
    # If no Symbol column, assume it's a single stock and create one
    merged_data['Symbol'] = 'STOCK'
    symbols = ['STOCK']

❌ 'Symbol' column not found in data


Updated Cell 3: Fix Data Structure and Identify Symbols

In [6]:
# Check the first row - it contains 'AAPL' values which should be column names
print("First row values:")
print(merged_data.iloc[0])

# Check the second row to understand the structure
print("\nSecond row values:")
print(merged_data.iloc[1])

# The issue: First row has 'AAPL' instead of proper data, but actual data starts from row 1
# Let's fix this by using the first row as column names and then removing it
print("\nFixing data structure...")

# Create a new DataFrame with proper column names from the first row
fixed_columns = merged_data.iloc[0].values
print(f"Columns from first row: {fixed_columns}")

# Create new DataFrame with proper structure
fixed_data = merged_data.iloc[1:].copy()
fixed_data.columns = fixed_columns

# Reset index
fixed_data = fixed_data.reset_index(drop=True)

print(f"\nFixed data shape: {fixed_data.shape}")
print(f"Fixed columns: {list(fixed_data.columns)}")
print(f"\nFirst few rows of fixed data:")
print(fixed_data.head())

First row values:
Date        NaN
Close      AAPL
High       AAPL
Low        AAPL
Open       AAPL
Volume     AAPL
Symbol    STOCK
Name: 0, dtype: object

Second row values:
Date             2020-01-02
Close     72.46825408935547
High      72.52857392672578
Low       71.22325176244068
Open      71.47659213409148
Volume            135480400
Symbol                STOCK
Name: 1, dtype: object

Fixing data structure...
Columns from first row: [nan 'AAPL' 'AAPL' 'AAPL' 'AAPL' 'AAPL' 'STOCK']

Fixed data shape: (1006, 7)
Fixed columns: [nan, 'AAPL', 'AAPL', 'AAPL', 'AAPL', 'AAPL', 'STOCK']

First few rows of fixed data:
          NaN               AAPL               AAPL               AAPL  \
0  2020-01-02  72.46825408935547  72.52857392672578  71.22325176244068   
1  2020-01-03  71.76372528076172  72.52375386627762  71.53933722124737   
2  2020-01-06   72.3355484008789  72.37415398165305   70.6345318765685   
3  2020-01-07  71.99536895751953  72.60097528872254  71.77580380389517   
4  2020-0

In [7]:
# Check the structure
print("First row values:")
print(merged_data.iloc[0])

print("\nSecond row values:")
print(merged_data.iloc[1])

# The problem: The actual column names are in the first row, but they're mixed up
# Let's extract the proper column names and create a clean DataFrame

# Get the actual column names from the original file
original_columns = list(merged_data.columns)
print(f"\nOriginal columns from file: {original_columns}")

# The first row contains the actual data column names, but they're mixed with values
first_row_values = merged_data.iloc[0].values
print(f"First row values: {first_row_values}")

# Create proper column mapping
# The first row shows us what each column should be named
proper_columns = ['Date', 'Close', 'High', 'Low', 'Open', 'Volume', 'Symbol']
print(f"Proper column names: {proper_columns}")

# Create a clean DataFrame starting from row 1 (which has the actual data)
clean_data = merged_data.iloc[1:].copy()
clean_data.columns = proper_columns

# Reset index
clean_data = clean_data.reset_index(drop=True)

print(f"\nClean data shape: {clean_data.shape}")
print(f"Clean columns: {list(clean_data.columns)}")
print(f"\nFirst few rows of clean data:")
print(clean_data.head())

First row values:
Date        NaN
Close      AAPL
High       AAPL
Low        AAPL
Open       AAPL
Volume     AAPL
Symbol    STOCK
Name: 0, dtype: object

Second row values:
Date             2020-01-02
Close     72.46825408935547
High      72.52857392672578
Low       71.22325176244068
Open      71.47659213409148
Volume            135480400
Symbol                STOCK
Name: 1, dtype: object

Original columns from file: ['Date', 'Close', 'High', 'Low', 'Open', 'Volume', 'Symbol']
First row values: [nan 'AAPL' 'AAPL' 'AAPL' 'AAPL' 'AAPL' 'STOCK']
Proper column names: ['Date', 'Close', 'High', 'Low', 'Open', 'Volume', 'Symbol']

Clean data shape: (1006, 7)
Clean columns: ['Date', 'Close', 'High', 'Low', 'Open', 'Volume', 'Symbol']

First few rows of clean data:
         Date              Close               High                Low  \
0  2020-01-02  72.46825408935547  72.52857392672578  71.22325176244068   
1  2020-01-03  71.76372528076172  72.52375386627762  71.53933722124737   
2  2020-01-